# Product 상세 정보 수집

이전 단계에서 수집한 Product ID 리스트를 사용하여 각 상품의 상세 정보를 수집합니다.

## Parameters

In [ ]:
# Papermill parameters (DAG에서 전달받음)
env = "stg"
dt = "2024-08-07"
version_date = "20240807"
map_api_base_url = "https://api.map-stg.sktelecom.com"
map_api_key = None
gcs_bucket_name = "air-airflow-stg"

# 노트북 내부 설정
api_delay = 0.5  # API 호출 간격 (초)
max_retries = 3  # 최대 재시도 횟수
retry_delay = 2.0  # 재시도 간격 (초)

## 1. 라이브러리 및 설정 로드

In [ ]:
import requests
import json
from datetime import datetime
from typing import List, Dict, Any, Optional
import time
from pathlib import Path
from google.cloud import storage
import os

# 이전 단계에서 저장한 Product ID 리스트 GCS에서 읽어오기
gcs_path = f"product_id_list/{dt}/product_ids_{version_date}.json"

try:
    storage_client = storage.Client()
    bucket = storage_client.bucket(gcs_bucket_name)
    blob = bucket.blob(gcs_path)
    
    # GCS에서 JSON 데이터 다운로드
    json_data = blob.download_as_text()
    product_ids_data = json.loads(json_data)
    
    collected_product_ids = product_ids_data["collected_product_ids"]
    collection_summary = product_ids_data["collection_summary"]
    
    print(f"✅ GCS에서 Product ID 리스트 로드 완료: gs://{gcs_bucket_name}/{gcs_path}")
    print(f"📊 로드된 Product ID 개수: {len(collected_product_ids)}개")
    
except Exception as e:
    print(f"❌ GCS에서 Product ID 리스트 로드 실패: {str(e)}")
    # 테스트용 더미 데이터
    collected_product_ids = ["PROD001", "PROD002", "PROD003"]
    collection_summary = {"total_count": 3, "timestamp": datetime.now().isoformat()}
    print(f"📝 테스트용 더미 데이터 사용: {len(collected_product_ids)}개")

print(f"이전 단계에서 받은 Product ID 개수: {len(collected_product_ids)}")
print(f"이전 단계 요약: {collection_summary}")
print(f"Version Date: {version_date}")
print(f"API 호출 간격: {api_delay}초")

## 2. 상세 정보 수집 함수 정의

In [ ]:
def fetch_product_detail(product_id: str, headers: Dict[str, str]) -> Dict[str, Any]:
    """단일 상품의 상세 정보를 가져오는 함수 (재시도 로직 포함, 실패 시 예외 발생)"""
    url = f"{map_api_base_url}/product-meta/basic-plan_847de97c-96c0-49b1-b3b1-900ddc587e54/product-info"
    if env == "prd":
        url = f"{map_api_base_url}/product-meta/basic-plan_a9af95d9-1930-4148-9c7b-7c64511d3442/product-info"

    params = {"legacyId": product_id}
    
    for attempt in range(max_retries + 1):
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
            )
            
            if response.status_code == 200:
                return response.json()
            elif response.status_code in [500, 502, 503, 504]:  # 서버 오류는 재시도
                if attempt < max_retries:
                    print(f"⏳ Product {product_id} - HTTP {response.status_code}, 재시도 {attempt+1}/{max_retries}")
                    time.sleep(retry_delay)
                    continue
                else:
                    raise Exception(f"Product {product_id} - HTTP {response.status_code} (재시도 {max_retries}회 모두 실패)")
            else:  # 4xx 오류도 실패로 처리
                raise Exception(f"Product {product_id} - HTTP {response.status_code}: {response.text}")
                
        except requests.exceptions.RequestException as e:
            if attempt < max_retries:
                print(f"⏳ Product {product_id} - 네트워크 오류, 재시도 {attempt+1}/{max_retries}: {str(e)}")
                time.sleep(retry_delay)
                continue
            else:
                raise Exception(f"Product {product_id} - 네트워크 오류 (재시도 {max_retries}회 모두 실패): {str(e)}")
    
    raise Exception(f"Product {product_id} - 알 수 없는 오류")

# API 헤더 설정
headers = {
    "Content-Type": "application/json",
    "x-apim-key": map_api_key
}

print("상세 정보 수집 함수 준비 완료 (재시도 로직 포함, 실패 시 task 중단)")

## 3. 상품 상세 정보 수집 시작

In [ ]:
if not collected_product_ids:
    raise ValueError("이전 단계에서 Product ID가 전달되지 않았습니다.")

print(f"총 {len(collected_product_ids)}개 상품 상세 정보 순차 수집 시작")
print(f"API 호출 간격: {api_delay}초, 재시도: {max_retries}회")
print("⚠️ 모든 상품이 성공적으로 수집되어야 합니다. 하나라도 실패하면 task가 중단됩니다.")
start_time = datetime.now()

# 결과 저장 변수 (모두 성공할 것이므로 failed_products 제거)
successful_results = []

# 순차적으로 API 호출 (한 건이라도 실패하면 예외 발생)
for i, product_id in enumerate(collected_product_ids):
    print(f"진행률: {i+1}/{len(collected_product_ids)} ({(i+1)/len(collected_product_ids)*100:.1f}%) - {product_id}")
    
    # fetch_product_detail이 실패하면 예외를 발생시키므로 try-catch 불필요
    result = fetch_product_detail(product_id, headers)
    successful_results.append(result)
    
    # API 호출 간격 (마지막 호출 후에는 대기하지 않음)
    if i < len(collected_product_ids) - 1:
        time.sleep(api_delay)
    
    # 10개마다 중간 결과 출력
    if (i + 1) % 10 == 0:
        print(f"  → 중간 결과: {len(successful_results)}개 성공")

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\n✅ 전체 수집 완료! 소요시간: {duration:.2f}초 ({duration/60:.1f}분)")
print(f"📊 성공: {len(successful_results)}개 (100%)")

## 4. 결과 분석 및 요약

In [ ]:
# 성공률 계산 (모두 성공이므로 100%)
total_attempted = len(collected_product_ids)
success_count = len(successful_results)
success_rate = 100.0  # 모두 성공

# 수집 결과 요약
collection_result = {
    "timestamp": datetime.now().isoformat(),
    "version_date": version_date,
    "collection_stats": {
        "total_product_ids_received": len(collected_product_ids),
        "total_attempted": total_attempted,
        "successful_collections": success_count,
        "success_rate_percent": success_rate,
        "duration_seconds": round(duration, 2)
    }
}

print("\n=== 상세 정보 수집 결과 요약 ===")
for key, value in collection_result["collection_stats"].items():
    print(f"{key}: {value}")

# 수집된 데이터 샘플 확인
if successful_results:
    print(f"\n첫 번째 수집 데이터 구조: {list(successful_results[0].keys())}")
    
    # managementInfo가 있는지 확인 (다음 단계 비교용)
    if 'managementInfo' in successful_results[0]:
        mgmt_info = successful_results[0]['managementInfo']
        print(f"managementInfo 구조: {list(mgmt_info.keys()) if isinstance(mgmt_info, dict) else type(mgmt_info)}")
        
        if isinstance(mgmt_info, dict) and 'mappedProductCode' in mgmt_info:
            mapped_code = mgmt_info['mappedProductCode']
            print(f"mappedProductCode 구조: {list(mapped_code.keys()) if isinstance(mapped_code, dict) else type(mapped_code)}")

## 5. GCS에 결과 저장 (mobile_plan_info_YYYYMMDD.json)

In [ ]:
# 파일명 및 경로 생성
if not version_date:
    version_date = datetime.now().strftime("%Y%m%d")

filename = f"mobile_plan_info_{version_date}.json"
gcs_path = f"product_meta_raw/{dt}/{filename}"

# 메타데이터와 함께 저장할 데이터 구성
final_data = {
    "metadata": {
        "created_at": datetime.now().isoformat(),
        "version_date": version_date,
        "collection_date": dt,
        "total_products": len(successful_results),
        "collection_duration_seconds": duration,
        "source": "MAP API - product-info endpoint"
    },
    "result_list": successful_results  # 기존 형식과 동일하게 result_list로 저장
}

# GCS에 업로드
try:
    storage_client = storage.Client()
    bucket = storage_client.bucket(gcs_bucket_name)
    blob = bucket.blob(gcs_path)
    
    # JSON 데이터를 문자열로 변환 후 업로드
    json_data = json.dumps(final_data, ensure_ascii=False, indent=2)
    blob.upload_from_string(json_data, content_type='application/json')
    
    gcs_full_path = f"gs://{gcs_bucket_name}/{gcs_path}"
    
    print(f"\n✅ GCS 저장 완료: {gcs_full_path}")
    print(f"📁 파일 크기: {len(json_data) / (1024*1024):.2f} MB")
    print(f"📊 저장된 상품 개수: {len(successful_results)}개")
    
except Exception as e:
    print(f"❌ GCS 저장 실패: {str(e)}")
    raise

# 다음 단계로 전달할 변수들 (XCom)
saved_file_path = gcs_full_path  # GCS 경로
product_collection_result = {
    "total_products_collected": len(successful_results),
    "error_count": 0,  # 항상 0
    "success_rate": 100.0  # 항상 100%
}
total_products_collected = len(successful_results)

print(f"\n📤 다음 단계로 전달할 정보:")
print(f"- 저장된 파일: {saved_file_path}")
print(f"- 수집된 상품 수: {total_products_collected}")
print(f"- 성공률: 100%")

## 완료

상품 상세 정보 수집이 완료되었습니다. 다음 단계에서 이전 데이터와 비교하여 변경사항을 감지합니다.